In [4]:
import sqlite3
print(sqlite3.sqlite_version)

3.50.4


In [5]:
#01:LEFT JOIN - sab customers  dikhao, chahe unka koi order ho ya na ho:

In [16]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT c."Customer ID", c.Country, o.Invoice, o.OrderValue
    FROM customers c
    LEFT JOIN orders o ON c."Customer ID" = o."Customer ID"
""", conn)
print(q1.shape)
print(q1['Invoice'].isnull().sum()) #kitne customers ka koi order match nahi hua.

(45028, 4)
0


In [7]:
#02:Real LEFT JOIN use-case -- Countries table banao jisme sab countries ho, kuch customers wale kuch na ho:

In [8]:
#ek choti reference table banao manually (kuch extra countries jinke customers nahi hai)
countries_ref = pd.DataFrame({
    'Country': ['United Kingdom', 'Germany', 'France', 'Atlantis', 'Narnia'] #Atlantis/Narnia fictional - no match milega
})
countries_ref.to_sql('countries_ref', conn, if_exists='replace', index=False)


q2 = pd.read_sql("""
    SELECT r.Country, COUNT(c."Customer ID") as customer_count
    FROM countries_ref r
    LEFT JOIN customers c ON r.Country = c.Country
    GROUP BY r.Country
""", conn)
print(q2)

          Country  customer_count
0        Atlantis               0
1          France              95
2         Germany             107
3          Narnia               0
4  United Kingdom            5410


In [9]:
#03:RIGHT JOINN(agar SQLite version support karta hai):

In [10]:
q3 = pd.read_sql("""
SELECT r.Country, c."Customer ID"
FROM customers c
RIGHT JOIN countries_ref r ON c.Country = r.Country
""", conn)
print(q3)

             Country  Customer ID
0     United Kingdom      13085.0
1     United Kingdom      13078.0
2     United Kingdom      15362.0
3     United Kingdom      18102.0
4             France      12682.0
...              ...          ...
5609  United Kingdom      15520.0
5610  United Kingdom      13298.0
5611         Germany      12713.0
5612        Atlantis          NaN
5613          Narnia          NaN

[5614 rows x 2 columns]


In [11]:
#04 agar right join error de (purana SQLite version): bas table swap karke LEFT JOIN use karo- jo uapr 3 step me already dikhaya.

In [12]:
#INNER JOIN--> sirf match wale rows
#LEFT JOIN--> left table ke sab rows+match wale right rows(na mile to NULL)
#RIGHT JOIN--> right table ke sab rows + match wale left rows(na mile to NULL)

In [13]:
conn.close()